# Test the Saved Spam Mail Detector

This notebook loads the saved Naive Bayes model and TF-IDF vectorizer without retraining them.

In [1]:
import re
import joblib

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Load the same English stopwords used when the model was trained.
english_stopwords = set(stopwords.words("english"))

def preprocess_text(text):
    """
    Apply the same preprocessing used for model training.
    """

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in english_stopwords]

    return " ".join(tokens)

In [2]:
# Load the saved model and TF-IDF vectorizer.

model_path = "../models/spam_mail_detector.joblib"

loaded_artifacts = joblib.load(model_path)

loaded_model = loaded_artifacts["model"]
loaded_vectorizer = loaded_artifacts["vectorizer"]

print("Saved model loaded successfully.")
print("Loaded model type:", type(loaded_model).__name__)
print("Loaded vectorizer type:", type(loaded_vectorizer).__name__)

Saved model loaded successfully.
Loaded model type: MultinomialNB
Loaded vectorizer type: TfidfVectorizer


## Predict New Emails with the Saved Model

In [3]:
def predict_saved_email(email_text):
    """
    Predict spam or ham using the saved model file.
    """

    # Apply the same preprocessing used during training.
    cleaned_text = preprocess_text(email_text)

    # Convert the cleaned email into the saved TF-IDF feature format.
    email_tfidf = loaded_vectorizer.transform([cleaned_text])

    # Predict ham (0) or spam (1).
    predicted_label = loaded_model.predict(email_tfidf)[0]

    # Get the model's estimated spam probability.
    spam_probability = loaded_model.predict_proba(email_tfidf)[0][1]

    prediction = "Spam" if predicted_label == 1 else "Ham"

    print("Prediction:", prediction)
    print(f"Estimated spam probability: {spam_probability:.2%}")

In [4]:
test_email = """
Congratulations! You have been selected to receive a free reward.
Click the link now to claim your prize.
"""

predict_saved_email(test_email)

Prediction: Spam
Estimated spam probability: 71.10%


In [5]:
# Test the saved model with one spam-style and one ham-style email.

spam_example = """
Congratulations! You have won a free reward.
Click now to claim your prize.
"""

ham_example = """
Hi team,

Please review the project agenda before tomorrow's meeting.

Thanks,
Aisha
"""

print("Spam-style email result:")
predict_saved_email(spam_example)

print("\nHam-style email result:")
predict_saved_email(ham_example)

Spam-style email result:
Prediction: Spam
Estimated spam probability: 66.33%

Ham-style email result:
Prediction: Ham
Estimated spam probability: 0.25%
